TEAMMATE STUDY GUIDE (Refer to Lecture 2 - Data Preparation):
1. Data Leakage Warning: We MUST calculate the median income (renta) from the TRAIN data 
   and use that exact same number to fill missing values in the TEST data. If we calculate 
   a new median for the test data, it is called "Data Leakage" and we will fail the lab.
2. 3-Sigma Rule: Instead of deleting million-dollar incomes, we cap them at 3 Standard 
   Deviations from the mean to remove extreme outliers without losing customer rows.

In [2]:
import pandas as pd
import numpy as np
import gc
import os

In [3]:
DATASET_DIR = "/home/gq11/Documents/USTH/ML2/Project/dataset"

# Product columns
PRODUCT_COLS = [
    'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1',
    'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
    'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
    'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1',
    'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1',
    'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1',
    'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1',
    'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1'
]

# Category columns
CATEGORY_COLS = [
    'sexo', 'tiprel_1mes', 'indrel_1mes',
    'indext', 'canal_entrada',
    'nomprov', 'segmento'
]

In [ ]:
def process_data(
    df,
    dataset_name,
    median_renta=None,
    median_age=None,
    province_income_map=None
):
    
    print(f"\n--- Processing {dataset_name} ---")
    
    # clean invalid
    df = df.replace([' NA', 'NA', ' ', ''],np.nan)

    # [Lecture 2] Deduplication
    before_rows = len(df)
    df = df.drop_duplicates()
    
    # date
    df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])
    df['fecha_alta'] = pd.to_datetime(df['fecha_alta'],errors='coerce')
    
    # age [18-100]: median
    df['age'] = pd.to_numeric(df['age'],errors='coerce')
    if median_age is None:
        median_age = df['age'].median()
    df['age'] = (df['age'].clip(lower=18, upper=100).fillna(median_age).astype('int8'))
    
    # antiguedad: thâm niên >
    df['antiguedad'] = pd.to_numeric(df['antiguedad'],errors='coerce')
    df['antiguedad'] = (df['antiguedad'].clip(lower=0).fillna(0).astype('int16'))

    # renta (income): median theo thành phố
    df['renta'] = pd.to_numeric(df['renta'],errors='coerce')
    if median_renta is None:
        median_renta = df['renta'].median()
    if province_income_map is not None:
        df['renta'] = df['renta'].fillna(df['nomprov'].map(province_income_map))
    df['renta'] = df['renta'].fillna(median_renta)
    
    # [Lecture 2] Outlier Capping (3-Sigma Rule for Renta, Logical Bounds for Age)
    upper_income = (df['renta'].mean() + 3 * df['renta'].std())
    df['renta'] = (df['renta'].clip(upper=upper_income).astype('float32'))
    
    # product: fill 0
    df[PRODUCT_COLS] = (df[PRODUCT_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).astype('int8'))
    
    # categorical
    for col in CATEGORY_COLS:
        if col in df.columns:
            mode_val = df[col].mode()
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val[0])
            df[col] = df[col].astype('category')
    
    # other
    binary_cols = ['ind_nuevo','ind_actividad_cliente']
    for col in binary_cols:
        if col in df.columns:
            df[col] = (
                pd.to_numeric(
                    df[col],
                    errors='coerce'
                )
                .fillna(0)
                .astype('int8')
            )
    if 'cod_prov' in df.columns:
        df['cod_prov'] = (pd.to_numeric(df['cod_prov'],errors='coerce').fillna(0).astype('int16'))
    
    # Verification
    print("\nMissing Values After Cleaning:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print(f"\nFinal Shape: {df.shape}")
    

    
    return df, median_renta, median_age


In [5]:
# Execute Pipeline
train_df = pd.read_parquet(os.path.join(DATASET_DIR,"train_optimized.parquet"))

# Province median from TRAIN ONLY
province_income_map = (train_df.groupby('nomprov')['renta'].median())

train_df, m_renta, m_age = process_data(train_df,"TRAINING DATA",province_income_map=province_income_map)

test_df = pd.read_parquet(os.path.join(DATASET_DIR,"test_optimized.parquet"))

test_df, _, _ = process_data(test_df, "TESTING DATA",m_renta,m_age,province_income_map)


--- Processing TRAINING DATA ---

Missing Values After Cleaning:
ind_empleado          27734
pais_residencia       27734
fecha_alta            27734
indrel                27734
ult_fec_cli_1t     12692933
indresi               27734
conyuemp           12714162
indfall               27734
tipodom               27735
dtype: int64

Final Shape: (12715856, 48)

--- Processing TESTING DATA ---

Missing Values After Cleaning:
ult_fec_cli_1t    929583
conyuemp          931339
dtype: int64

Final Shape: (931453, 48)


In [6]:
# =========================================================
# SAVE
# =========================================================

train_df.to_parquet(
    os.path.join(
        DATASET_DIR,
        "train_cleaned.parquet"
    ),
    engine='pyarrow'
)

test_df.to_parquet(
    os.path.join(
        DATASET_DIR,
        "test_cleaned.parquet"
    ),
    engine='pyarrow'
)

# =========================================================
# CLEAN MEMORY
# =========================================================

del train_df
del test_df

gc.collect()

print("\nPipeline Completed Successfully.")


Pipeline Completed Successfully.
